In [ ]:
import unittest
# import os
# import boto3


In [ ]:
# Configuração de leitura no Athena a partir de um job Python shell do Glue Studio.
# aws_region = os.environ["AWS_REGION"]
# athena_database = os.environ["ATHENA_DATABASE"]
# athena_table = os.environ["ATHENA_TABLE"]
# athena_results_s3 = os.environ["ATHENA_RESULTS_S3"]
# athena = boto3.client("athena", region_name=aws_region)  # usa a função IAM do job
# resposta = athena.start_query_execution(
#     QueryString=f"SELECT * FROM {athena_table}",
#     QueryExecutionContext={"Database": athena_database},
#     ResultConfiguration={"OutputLocation": athena_results_s3},
# )
# query_execution_id = resposta["QueryExecutionId"]
# A leitura requer aguardar SUCCEEDED e paginar get_query_results.


In [ ]:
class Estoque:

    def __init__(self):
        self._produtos = {}
        self._registros = {}

    @staticmethod
    def _validar_id(valor, nome):
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError(f"{nome} deve ser um texto não vazio")

    @staticmethod
    def _validar_quantidade(quantidade):
        # bool é subtipo de int em Python, mas não representa unidades.
        if type(quantidade) is not int or quantidade <= 0:
            raise ValueError("quantidade deve ser um inteiro positivo")

    def _produto_existente(self, produto_id):
        self._validar_id(produto_id, "produto_id")
        if produto_id not in self._produtos:
            raise ValueError(f"produto não encontrado: {produto_id}")

    def consultar(self, produto_id):
        self._produto_existente(produto_id)
        ativos = (
            registro for registro in self._registros.values()
            if registro["produto_id"] == produto_id and registro["ativo"]
        )
        reservada = 0
        bloqueada = 0
        for registro in ativos:
            if registro["tipo"] == "reserva":
                reservada += registro["quantidade"]
            else:
                bloqueada += registro["quantidade"]
        total = self._produtos[produto_id]
        return {
            "produto_id": produto_id,
            "total": total,
            "disponivel": total - reservada - bloqueada,
            "reservada": reservada,
            "bloqueada": bloqueada,
        }

    def consultar_registro(self, registro_id):
        self._validar_id(registro_id, "registro_id")
        if registro_id not in self._registros:
            raise ValueError(f"registro não encontrado: {registro_id}")
        return self._registros[registro_id].copy()

    def entrada(self, produto_id, quantidade):
        self._validar_id(produto_id, "produto_id")
        self._validar_quantidade(quantidade)
        # A primeira entrada cadastra o produto com estoque inicial zero.
        self._produtos[produto_id] = self._produtos.get(produto_id, 0) + quantidade
        return self.consultar(produto_id)

    def saida(self, produto_id, quantidade):
        self._validar_quantidade(quantidade)
        saldo = self.consultar(produto_id)
        if quantidade > saldo["disponivel"]:
            raise ValueError("quantidade disponível insuficiente")
        self._produtos[produto_id] -= quantidade
        return self.consultar(produto_id)

    def _registrar(self, tipo, produto_id, solicitacao_id, registro_id, quantidade):
        self._validar_id(solicitacao_id, "solicitacao_id")
        self._validar_id(registro_id, "registro_id")
        self._validar_quantidade(quantidade)
        saldo = self.consultar(produto_id)
        if registro_id in self._registros:
            raise ValueError(f"registro_id já utilizado: {registro_id}")
        if quantidade > saldo["disponivel"]:
            raise ValueError("quantidade disponível insuficiente")
        # Só altera o estado após todas as validações.
        self._registros[registro_id] = {
            "registro_id": registro_id,
            "tipo": tipo,
            "produto_id": produto_id,
            "solicitacao_id": solicitacao_id,
            "quantidade": quantidade,
            "ativo": True,
        }
        return self.consultar(produto_id)

    def reservar(self, produto_id, solicitacao_id, reserva_id, quantidade):
        return self._registrar("reserva", produto_id, solicitacao_id, reserva_id, quantidade)

    def bloquear(self, produto_id, solicitacao_id, bloqueio_id, quantidade):
        return self._registrar("bloqueio", produto_id, solicitacao_id, bloqueio_id, quantidade)

    def liberar(self, registro_id):
        registro = self.consultar_registro(registro_id)
        if not registro["ativo"]:
            raise ValueError(f"registro já liberado: {registro_id}")
        self._registros[registro_id]["ativo"] = False
        return self.consultar(registro["produto_id"])

In [ ]:
class TesteEstoque(unittest.TestCase):

    def setUp(self):
        self.estoque = Estoque()
        self.estoque.entrada("P1", 10)

    def test_entrada_e_saida(self):
        self.estoque.entrada("P1", 5)
        saldo = self.estoque.saida("P1", 4)
        self.assertEqual((saldo["total"], saldo["disponivel"]), (11, 11))

    def test_reservas_e_bloqueios_simultaneos(self):
        self.estoque.reservar("P1", "PED-1", "R1", 3)
        self.estoque.reservar("P1", "PED-2", "R2", 2)
        saldo = self.estoque.bloquear("P1", "PED-3", "B1", 1)
        self.assertEqual((saldo["total"], saldo["disponivel"], saldo["reservada"], saldo["bloqueada"]), (10, 4, 5, 1))
        self.assertEqual(self.estoque.consultar_registro("R2")["solicitacao_id"], "PED-2")

    def test_liberar_reserva_e_bloqueio(self):
        self.estoque.reservar("P1", "PED-1", "R1", 3)
        self.estoque.bloquear("P1", "PED-2", "B1", 2)
        self.estoque.liberar("R1")
        saldo = self.estoque.liberar("B1")
        self.assertEqual((saldo["disponivel"], saldo["reservada"], saldo["bloqueada"]), (10, 0, 0))
        self.assertFalse(self.estoque.consultar_registro("R1")["ativo"])

    def test_saldo_insuficiente_nao_altera_estoque(self):
        self.estoque.reservar("P1", "PED-1", "R1", 8)
        anterior = self.estoque.consultar("P1")
        for operacao in (
            lambda: self.estoque.saida("P1", 3),
            lambda: self.estoque.reservar("P1", "PED-2", "R2", 3),
            lambda: self.estoque.bloquear("P1", "PED-3", "B1", 3),
        ):
            with self.assertRaisesRegex(ValueError, "insuficiente"):
                operacao()
            self.assertEqual(self.estoque.consultar("P1"), anterior)

    def test_quantidade_invalida_nao_altera_estoque(self):
        anterior = self.estoque.consultar("P1")
        for quantidade in (0, -1, 1.5, True):
            with self.assertRaises(ValueError):
                self.estoque.entrada("P1", quantidade)
            self.assertEqual(self.estoque.consultar("P1"), anterior)

    def test_identificadores_duplicados_e_liberacao_repetida(self):
        self.estoque.reservar("P1", "PED-1", "ID-1", 2)
        with self.assertRaises(ValueError):
            self.estoque.bloquear("P1", "PED-2", "ID-1", 1)
        self.estoque.liberar("ID-1")
        with self.assertRaises(ValueError):
            self.estoque.liberar("ID-1")
        with self.assertRaises(ValueError):
            self.estoque.reservar("P1", "PED-3", "ID-1", 1)

    def test_produtos_independentes(self):
        self.estoque.entrada("P2", 4)
        self.estoque.reservar("P1", "PED-1", "R1", 3)
        self.assertEqual(self.estoque.consultar("P2")["disponivel"], 4)

    def test_produto_ou_registro_inexistente(self):
        with self.assertRaises(ValueError):
            self.estoque.saida("P2", 1)
        with self.assertRaises(ValueError):
            self.estoque.consultar_registro("R-inexistente")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TesteEstoque)
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
if not resultado.wasSuccessful():
    raise AssertionError("Os testes do estoque falharam")